# Randomized Max-Margin Classifier

This notebook is intentionally self-contained. It does not import local utility files, so the statistical idea, simulation, Python functions, evaluation, plots, and exam translation are all visible in one place.

## What problem are we solving?

A max-margin classifier looks for a separating boundary that keeps the closest training points as far from the boundary as possible.

**Highest-probability exam pattern:** Implement randomized line search for a separable two-dimensional data set and connect it to the SVM margin idea.

## Assumptions, intuition, and theory

The geometric margin is the minimum distance from any training point to the separating line. Among perfectly separating lines, the max-margin line is expected to generalize better than a barely separating line.

## Python method notes and documentation

`np.sign` converts line scores to class predictions, point-to-line distance computes the margin, and `LinearSVC` gives a standard max-margin-style comparison.

Documentation links:

- [NumPy random generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [matplotlib pyplot](https://matplotlib.org/stable/api/pyplot_summary.html)
- [SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)
- [LinearSVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html)

## Fully self-contained worked example

Every code line below is commented so it is easy to scan under exam time pressure.

In [ ]:
import numpy as np  # Import numerical arrays and random-number tools.
import pandas as pd  # Import DataFrame tables for line summaries.
import matplotlib.pyplot as plt  # Import plotting tools for separating lines.
from sklearn.svm import LinearSVC  # Import linear support-vector classifier for comparison.
RANDOM_SEED = 123  # Store the reproducibility seed.
rng = np.random.default_rng(RANDOM_SEED)  # Create a reproducible random generator.
plt.style.use('default')  # Use a predictable plotting style.


In [ ]:
def make_separable_data(random_state=123):  # Define a clean linearly separable classification simulator.
    rng = np.random.default_rng(random_state)  # Create a reproducible random generator.
    X0 = rng.normal(loc=[-1.5, -1.5], scale=0.35, size=(35, 2))  # Generate class 0 points around the lower-left center.
    X1 = rng.normal(loc=[1.5, 1.5], scale=0.35, size=(35, 2))  # Generate class 1 points around the upper-right center.
    X = np.vstack([X0, X1])  # Stack the two class-specific arrays.
    y = np.array([0] * len(X0) + [1] * len(X1))  # Create binary class labels.
    return X, y  # Return predictors and labels.

def randomized_max_margin_classifier(X, y, n_lines=1000, make_plot=True, random_state=123):  # Define a randomized search for a separating line.
    rng = np.random.default_rng(random_state)  # Create a reproducible random generator.
    X = np.asarray(X)  # Convert predictors to a NumPy array.
    y_signed = np.where(np.asarray(y) == 1, 1, -1)  # Convert labels from 0/1 to -1/+1 signs.
    best_coef = None  # Initialize the best line coefficients.
    best_margin = -np.inf  # Initialize the best margin as impossible low value.
    for candidate_number in range(n_lines):  # Try many random candidate lines.
        coef = rng.normal(size=3)  # Draw random intercept and two slope coefficients.
        scores = coef[0] + coef[1] * X[:, 0] + coef[2] * X[:, 1]  # Compute signed line scores.
        if np.mean(np.sign(scores) == y_signed) < 0.5:  # Check whether the orientation is mostly backwards.
            coef = -coef  # Flip the line orientation.
            scores = -scores  # Flip the scores to match the new orientation.
        if not np.all(np.sign(scores) == y_signed):  # Skip lines that do not perfectly separate the data.
            continue  # Move to the next random line.
        distances = np.abs(scores) / np.sqrt(coef[1] ** 2 + coef[2] ** 2)  # Compute point-to-line distances.
        margin = np.min(distances)  # Define the margin as the closest-point distance.
        if margin > best_margin:  # Check whether this line improves the current best margin.
            best_margin = margin  # Store the improved margin.
            best_coef = coef  # Store the improved line coefficients.
    if make_plot and best_coef is not None:  # Plot only when a separating line was found.
        plt.figure(figsize=(6, 4))  # Create a figure for the max-margin line.
        plt.scatter(X[:, 0], X[:, 1], c=y, s=32, alpha=0.8)  # Plot points colored by class.
        x_grid = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 100)  # Create x values for the separating line.
        y_grid = -(best_coef[0] + best_coef[1] * x_grid) / best_coef[2]  # Convert line coefficients to y-values.
        plt.plot(x_grid, y_grid, color='red', label=f'random max margin={best_margin:.3f}')  # Plot the selected separating line.
        plt.xlabel('x1')  # Label the first predictor axis.
        plt.ylabel('x2')  # Label the second predictor axis.
        plt.title('Randomized max-margin separating line')  # Title the plot.
        plt.legend()  # Show the margin label.
        plt.show()  # Render the plot.
    return {'best_line': best_coef, 'margin': best_margin}  # Return the best line and margin.


In [ ]:
X, y = make_separable_data(random_state=RANDOM_SEED)  # Simulate linearly separable data.
random_result = randomized_max_margin_classifier(X, y, n_lines=5000, make_plot=True, random_state=RANDOM_SEED)  # Search for a separating line with large margin.
svm_model = LinearSVC(C=1e6, max_iter=20000, random_state=RANDOM_SEED)  # Create a near-hard-margin linear SVM for comparison.
svm_model.fit(X, y)  # Fit the linear SVM.
print(random_result)  # Print the randomized search result.
print('LinearSVC coefficients:', svm_model.coef_, 'intercept:', svm_model.intercept_)  # Print the SVM separating boundary parameters.


## Exam-style problem prompt

A prompt asks you to explain or implement a max-margin idea in two dimensions. Generate many random separating lines, compute each margin, and keep the largest one.

## Adaptable code pattern

```python
# Step 1: Convert labels to -1 and +1.
# Step 2: Generate many random lines.
# Step 3: Keep only lines that classify every training point correctly.
# Step 4: Compute the closest point-to-line distance for each valid line.
# Step 5: Return the line with the largest minimum distance.
# Step 6: Explain the connection to support vectors and SVMs.
```

## What to remember for the exam

The key idea is not randomness; it is margin. The randomized search is just an exam-friendly way to approximate the max-margin concept.